# Dobot + Laser Experiment Notebook

Manual notebook organized to match `experiment.py`. Use the full experiment cell for the normal workflow; the manual cells are for checking hardware state and recovery.

## 1. Imports And Config

In [ ]:
import config
from Dobot import (
    connect_robot,
    has_robot_error,
    move_linear_point,
    prepare_robot,
    return_to_pose,
    send_do_pulse,
    start_feedback,
    turn_do_off,
)
from Laser import connect_laser
from experiment import (
    build_step_target,
    calculate_step_count,
    initialize_laser_from_config,
    run_experiment,
)
from time import sleep

dobot = None
laser = None
saved_start_pose = None
records = None

In [ ]:
print("Dobot IP:", config.DOBOT_IP)
print("Speed ratio:", config.SPEED_RATIO)
print("Laser DLL path:", config.LASER_DLL_PATH)
print("Laser wavelength:", config.LASER_WAVELENGTH_NM, "nm")
print("Laser mode:", config.LASER_FIRE_MODE)
print("Step distance:", config.STEP_DISTANCE_MM, "mm")
print("Total distance:", config.TOTAL_DISTANCE_MM, "mm")
print("Step wait:", config.STEP_WAIT_SECONDS, "s")
print("Trigger DO:", config.TRIGGER_DO_INDEX)
print("Trigger pulse:", config.TRIGGER_PULSE_SECONDS, "s")

## 2. Connect Hardware

In [ ]:
dobot = connect_robot(config.DOBOT_IP)
laser = connect_laser(config.LASER_DLL_PATH)
print("Robot and laser connected")

## 3. Prepare Robot And Laser

In [ ]:
if not prepare_robot(dobot, config.SPEED_RATIO):
    raise RuntimeError("Robot prepare failed")

initialize_laser_from_config(laser)

if has_robot_error(dobot):
    raise RuntimeError("Robot has active errors")

start_feedback(dobot)
sleep(1)
print("Robot and laser ready")

## 4. Save Original Pose

In [ ]:
saved_start_pose = dobot.GetCurrentPose()
step_count = calculate_step_count()

print("Saved start pose:", saved_start_pose)
print("Direction: -Y")
print("Step count:", step_count)

## 5. Optional Manual Checks

In [ ]:
current_pose = dobot.GetCurrentPose()
print("Current pose:", current_pose)

In [ ]:
do_on_result, do_off_result = send_do_pulse(
    dobot,
    config.TRIGGER_DO_INDEX,
    config.TRIGGER_PULSE_SECONDS,
)
print("DO pulse:", do_on_result, do_off_result)

In [ ]:
turn_do_off(dobot, config.TRIGGER_DO_INDEX)

## 6. Optional One Step Preview

In [ ]:
if saved_start_pose is None:
    raise RuntimeError("Run Save Original Pose first")

target_pose = build_step_target(saved_start_pose, 1)
print("Preview target pose:", target_pose)
# Uncomment to move one step manually.
# move_linear_point(dobot, target_pose, config.SPEED_RATIO)

## 7. Run Full Experiment

In [ ]:
records = run_experiment(require_confirm=True)
records

## 8. View Records

In [ ]:
do_records = records.get("do_records", []) if isinstance(records, dict) else records
endpoint_records = records.get("endpoint_records", []) if isinstance(records, dict) else []

print("DO records:", len(do_records))
print("Endpoint records:", len(endpoint_records))
print("Endpoints:", endpoint_records)

In [ ]:
try:
    import pandas as pd
    display(pd.DataFrame(do_records))
    display(pd.DataFrame(endpoint_records))
except ImportError:
    print(do_records)
    print(endpoint_records)

## 9. Recovery And Shutdown

In [ ]:
if laser is not None:
    laser.stop_safely()

if dobot is not None and saved_start_pose is not None:
    return_to_pose(
        dobot,
        saved_start_pose,
        config.SPEED_RATIO,
        do_indexes=[config.TRIGGER_DO_INDEX],
    )

In [ ]:
if laser is not None:
    laser.close()
    laser = None
print("Laser closed")

## 10. Laser Manual Reference

In [ ]:
laser = connect_laser(config.LASER_DLL_PATH)
initialize_laser_from_config(laser)
laser.set_wavelength(config.LASER_WAVELENGTH_NM)

In [ ]:
laser.run()
# laser stays running until you run laser.stop().

In [ ]:
laser.stop()

In [ ]:
laser.close()
laser = None